### Settings KeplerGl

In [42]:
from keplergl import KeplerGl

config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 47.3769,
            "longitude": 8.47,
            "zoom": 10,
            "bearing": 0,
            "pitch": 0
        },
        "visState": {
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "Trajectories": [
                            {"name": "length (km)"},
                            {"name": "time (min)"}
                        ]
                    },
                    "enabled": True
                }
            }
        }
    }
}

map_1 = KeplerGl(height=700, config=config)

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


### Open JSON

In [43]:
import json

# Load full trajectory linestrings from JSON
json_path = '../WorldMove/output/trajectories_lines.json'
with open(json_path, 'r') as f:
    trajectories = json.load(f)

### Calculate Drivingdistance in km

In [44]:
import math
import json
import os


def haversine(p0, p1):
    """Calculate geodesic distance between two lon/lat points in meters."""
    lon1, lat1 = p0
    lon2, lat2 = p1
    r = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(min(1, math.sqrt(a)))


def compute_feature_length(feature):
    """Compute the length of a LineString feature using haversine distance, returns km."""
    geometry = feature.get('geometry', {})
    if geometry.get('type') != 'LineString':
        return None
    coords = geometry.get('coordinates', [])
    total = 0.0
    for p0, p1 in zip(coords, coords[1:]):
        total += haversine(p0, p1)
    return round(total / 1000, 2)  # Return in kilometers


# Compute lengths for all features in the loaded trajectories
print(f"Computing lengths for {len(trajectories.get('features', []))} features...")
lengths = []
for feature in trajectories.get('features', []):
    length_km = compute_feature_length(feature)
    feature.setdefault('properties', {})['length (km)'] = length_km
    feature['properties']['length'] = length_km
    lengths.append(length_km)


length_20 = [l for l in lengths if l < 20]
if length_20:
    print(f"  LineStrings < 20km: {len(length_20)}")

# Create a subset dataset for features shorter than 20km
short_features = [
    feature
    for feature in trajectories.get('features', [])
    if feature.get('properties', {}).get('length (km)') is not None
    and feature['properties']['length (km)'] < 20
]

short_trajectories = {
    key: trajectories[key]
    for key in ('type', 'name', 'crs')
    if key in trajectories
}
short_trajectories['features'] = short_features

# Save filtered trajectories to new JSON file
output_dir = os.path.join(os.getcwd(), '..', 'Visualisierung', 'output_vis')
filtered_json_path = os.path.join(output_dir, 'trajectories_lines_filtered_20km.json')
os.makedirs(output_dir, exist_ok=True)

with open(filtered_json_path, 'w', encoding='utf-8') as f:
    json.dump(short_trajectories, f, ensure_ascii=False, indent=2)

print(f"\nSaved filtered trajectories to: {filtered_json_path}")
print(f"Features in filtered file: {len(short_features)}")

# Print summary
valid_lengths = [l for l in lengths if l is not None]
print(f"\nComputation complete!")
print(f"  Features processed: {len(lengths)}")
print(f"  LineStrings with lengths: {len(valid_lengths)}")
if valid_lengths:
    print(f"  min length (km): {min(valid_lengths):.2f}")
    print(f"  max length (km): {max(valid_lengths):.2f}")
    print(f"  mean length (km): {sum(valid_lengths) / len(valid_lengths):.2f}")

Computing lengths for 200 features...
  LineStrings < 20km: 114

Saved filtered trajectories to: c:\Users\adrie\github\4040_GP2\Hackathon\BuildABridge\Visualisierung\..\Visualisierung\output_vis\trajectories_lines_filtered_20km.json
Features in filtered file: 114

Computation complete!
  Features processed: 200
  LineStrings with lengths: 200
  min length (km): 0.00
  max length (km): 189.53
  mean length (km): 29.58


### Calculate Drivingtime

In [45]:
speed = 15  # km/h

time = []

for feature in trajectories.get('features', []):

    # Länge des aktuellen Features holen
    length_km = feature.get('properties', {}).get('length (km)', None)

    if length_km is not None:

        # Zeit berechnen
        calc_time = length_km / speed * 60

        # In properties speichern
        feature.setdefault('properties', {})['time (min)'] = round(calc_time, 2)

        # In Liste speichern
        time.append(round(calc_time, 2))

print(time)

[50.44, 45.6, 0.0, 226.36, 40.6, 83.64, 47.64, 54.28, 21.08, 104.64, 62.92, 0.0, 29.92, 7.36, 153.64, 0.0, 227.76, 34.28, 42.04, 240.24, 27.0, 351.32, 38.44, 17.96, 241.64, 20.44, 109.32, 35.04, 64.6, 322.76, 45.72, 17.0, 75.8, 67.32, 23.88, 256.4, 145.92, 119.4, 0.0, 87.76, 437.64, 48.64, 287.64, 34.04, 39.24, 370.64, 128.28, 135.92, 104.92, 0.0, 248.48, 0.0, 630.72, 46.16, 14.2, 206.88, 125.6, 228.4, 78.72, 387.52, 37.32, 36.16, 55.76, 76.52, 115.64, 4.52, 257.6, 558.24, 0.0, 399.52, 173.4, 61.48, 8.76, 32.92, 82.8, 139.48, 0.0, 0.0, 416.08, 118.24, 0.0, 301.16, 19.6, 203.12, 403.96, 54.28, 15.56, 127.44, 79.8, 303.88, 0.0, 0.0, 46.56, 42.08, 20.4, 98.16, 53.36, 128.6, 0.0, 67.44, 166.24, 79.96, 55.92, 16.24, 290.6, 616.96, 296.88, 87.96, 71.72, 84.76, 65.56, 8.28, 105.44, 82.48, 194.72, 111.84, 50.48, 132.76, 117.52, 84.16, 60.24, 1.56, 58.68, 43.28, 151.36, 30.72, 226.44, 56.16, 32.44, 502.92, 161.76, 69.52, 181.6, 9.44, 72.92, 178.48, 143.44, 153.84, 56.24, 87.56, 41.28, 678.6, 41

### Create new JSON with Distance in km and Time

In [46]:
import json
import os

output_path = os.path.join(os.getcwd(), 'output_vis', 'trajectories_lines_filtered_20km_with_time.json')

if 'filtered_trajectories' not in globals():
    filtered_json_path = os.path.join('..', 'Visualisierung', 'output_vis', 'trajectories_lines_filtered_20km.json')
    with open(filtered_json_path, 'r', encoding='utf-8') as f:
        filtered_trajectories = json.load(f)

for feature in filtered_trajectories.get('features', []):
    props = feature.setdefault('properties', {})
    if 'time (min)' not in props:
        length_km = props.get('length (km)')
        if length_km is not None and 'speed' in globals():
            props['time (min)'] = round(length_km / speed * 60, 2)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(filtered_trajectories, f, ensure_ascii=False, indent=2)

print(f"Saved updated trajectory JSON to: {output_path}")

Saved updated trajectory JSON to: c:\Users\adrie\github\4040_GP2\Hackathon\BuildABridge\Visualisierung\output_vis\trajectories_lines_filtered_20km_with_time.json


In [47]:
# Load the filtered trajectories from the newly created file
filtered_json_path = '../Visualisierung/output_vis/trajectories_lines_filtered_20km_with_time.json'
with open(filtered_json_path, 'r') as f:
    filtered_trajectories = json.load(f)

# Add the filtered GeoJSON data to the map
map_1.add_data(data=filtered_trajectories, name='Trajectories')
print(f"Added {len(filtered_trajectories.get('features', []))} filtered features to map")

Added 114 filtered features to map


In [48]:
# Save map as HTML file
import os

# Define output folder (output_vis in the current Visualisierung directory)
output_dir = os.path.join(os.getcwd(), 'output_vis')

# Create folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save HTML file
html_path = os.path.join(output_dir, 'trajectories_map.html')

map_1.save_to_html(file_name=html_path)

Map saved to c:\Users\adrie\github\4040_GP2\Hackathon\BuildABridge\Visualisierung\output_vis\trajectories_map.html!
